# Assessment — Dados e SQL

Este Assessment é resolvido **inteiramente neste notebook do Deepnote**, usando **blocos SQL nativos**. Ele tem duas partes:

- **Parte A — Queries (12 exercícios):** DDL, manipulação de dados e consultas com JOINs e agregações, sobre dois bancos distintos.
- **Parte B — Projeto de Modelagem (4 exercícios):** análise de entidades, relacionamentos, normalização e implementação física de um schema a partir de um caso de negócio. Resolvida nas células finais deste mesmo notebook.

### Como os dados entram: importação de CSV

Os dados **não** vêm de scripts `CREATE TABLE` prontos, mas em um conjunto de arquivos **CSV**. A importação cria tabelas com tipos inferidos e **sem restrições** (sem chaves, sem `NOT NULL`, sem `FOREIGN KEY`) — exatamente como dados crus chegam no mundo real. Parte dos exercícios consiste justamente em transformar esses dados crus em um schema bem projetado.

### Célula de importação = botão de reset

A célula de importação de cada banco usa `CREATE OR REPLACE TABLE`. **Reexecutá-la a qualquer momento restaura o estado inicial** dos dados. Como vários exercícios modificam tabelas (`INSERT`, `UPDATE`, `DELETE`, `DROP`), trate cada exercício como se partisse do estado recém-importado: se um exercício anterior alterou os dados, **reexecute a célula de importação do banco antes de resolver um exercício de consulta**.

### Sobre datas

Todos os filtros temporais usam datas fixas no formato `'AAAA-MM-DD'`. **Não use `CURRENT_DATE`** — os enunciados pressupõem datas explícitas para resultados reproduzíveis.

---

# 🅐 Parte A — Queries

# Banco 1 — Biblioteca Municipal

## Arquivos CSV do Banco 1 (o professor disponibiliza no projeto)

**`autores.csv`**
```csv
id,nome,nacionalidade,ano_nascimento
1,Machado de Assis,Brasileira,1839
2,Clarice Lispector,Brasileira,1920
3,Jorge Amado,Brasileira,1912
4,Gabriel García Márquez,Colombiana,1927
5,Haruki Murakami,Japonesa,1949
6,Chimamanda Adichie,Nigeriana,1977
```

**`livros.csv`**
```csv
id,titulo,autor_id,ano_publicacao,exemplares_disponiveis
1,Dom Casmurro,1,1899,3
2,Memórias Póstumas de Brás Cubas,1,1881,2
3,Quincas Borba,1,1891,1
4,A Hora da Estrela,2,1977,1
5,Água Viva,2,1973,2
6,Capitães da Areia,3,1937,4
7,"Gabriela, Cravo e Canela",3,1958,3
8,Cem Anos de Solidão,4,1967,2
9,O Amor nos Tempos do Cólera,4,1985,1
10,Norwegian Wood,5,1987,2
11,Americanah,6,2013,3
```

**`membros.csv`**
```csv
id,nome,email,data_inscricao
1,Ana Silva,ana@biblio.org,2023-01-15
2,Bruno Lima,bruno@biblio.org,2023-03-22
3,Carla Reis,carla@biblio.org,2024-02-10
4,Davi Castro,davi@biblio.org,2024-05-01
5,Edu Martins,edu@biblio.org,2024-08-12
```

**`emprestimos.csv`** (campos vazios em `data_devolucao_real` representam empréstimos não devolvidos → `NULL`)
```csv
id,livro_id,membro_id,data_emprestimo,data_devolucao_prevista,data_devolucao_real,valor_multa
1,1,1,2023-08-15,2023-08-29,2023-08-28,0
2,4,2,2023-11-10,2023-11-24,2023-12-01,5
3,6,1,2024-02-05,2024-02-19,2024-02-18,0
4,2,3,2026-02-10,2026-02-24,,15
5,8,4,2026-01-20,2026-02-03,,28
6,10,2,2026-03-01,2026-03-15,,10
7,5,5,2025-12-10,2025-12-24,2025-12-23,0
8,9,5,2026-05-01,2026-05-30,,0
9,11,4,2025-09-20,2025-10-04,2025-10-04,0
```


## Célula de importação do Banco 1 — *bloco SQL · execute antes de tudo*



> Após esta célula, as quatro tabelas existem com tipos inferidos, **mas sem chaves nem restrições** — você vai lidar com isso nos primeiros exercícios.

---


In [15]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

_dntk.execute_sql(
  '-- IMPORTAÇÃO / RESET — Banco 1: Biblioteca Municipal\nCREATE OR REPLACE TABLE autores (\n    id INTEGER PRIMARY KEY,\n    nome TEXT NOT NULL,\n    nacionalidade TEXT,\n    ano_nascimento INTEGER\n);\n\nINSERT INTO autores SELECT * FROM read_csv_auto(\'autores.csv\');CREATE OR REPLACE TABLE livros      AS SELECT * FROM read_csv_auto(\'livros.csv\');\nCREATE OR REPLACE TABLE membros     AS SELECT * FROM read_csv_auto(\'membros.csv\');\nCREATE OR REPLACE TABLE emprestimos AS SELECT * FROM read_csv_auto(\'emprestimos.csv\');\n',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)

,Count
0,9


### Exercício 1

A biblioteca decidiu cadastrar formalmente seus funcionários numa tabela **bem definida desde o início** (diferente das tabelas importadas do CSV).

**Tarefa.** Crie uma tabela `funcionarios` com restrições adequadas:

- `id`
- `nome`
- `email`
- `cargo`
- `data_admissao`

Insira os 4 funcionários abaixo e execute um `SELECT` ordenado por `data_admissao`.

| id | nome | email | cargo | data_admissao |
|---|---|---|---|---|
| 1 | Lúcia Andrade | lucia@biblio.org | Bibliotecária | 2022-04-10 |
| 2 | Renato Souza | renato@biblio.org | Atendente | 2023-09-01 |
| 3 | Marta Oliveira | marta@biblio.org | Bibliotecária | 2024-01-15 |
| 4 | Iago Pereira | iago@biblio.org | Auxiliar | 2024-11-20 |


---


In [1]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

_dntk.execute_sql(
  '-- Exercício 1\n-- Escreva sua solução SQL aqui.\nCREATE TABLE funcionarios (\n    id INTEGER PRIMARY KEY,\n    nome VARCHAR(100) NOT NULL,\n    email VARCHAR(100) NOT NULL UNIQUE,\n    cargo VARCHAR(50) NOT NULL,\n    data_admissao DATE NOT NULL\n);\n\nINSERT INTO funcionarios (id, nome, email, cargo, data_admissao) VALUES\n(1, \'Lúcia Andrade\', \'lucia@biblio.org\', \'Bibliotecária\', \'2022-04-10\'),\n(2, \'Renato Souza\', \'renato@biblio.org\', \'Atendente\', \'2023-09-01\'),\n(3, \'Marta Oliveira\', \'marta@biblio.org\', \'Bibliotecária\', \'2024-01-15\'),\n(4, \'Iago Pereira\', \'iago@biblio.org\', \'Auxiliar\', \'2024-11-20\');\n\nSELECT id, nome, email, cargo, data_admissao\nFROM funcionarios\nORDER BY data_admissao ASC;',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)

,id,nome,email,cargo,data_admissao
0,1,Lúcia Andrade,lucia@biblio.org,Bibliotecária,2022-04-10
1,2,Renato Souza,renato@biblio.org,Atendente,2023-09-01
2,3,Marta Oliveira,marta@biblio.org,Bibliotecária,2024-01-15
3,4,Iago Pereira,iago@biblio.org,Auxiliar,2024-11-20


### Exercício 2

A tabela `livros` veio do CSV **sem nenhuma restrição**: não há chave primária, nada impede um título nulo, e nada garante que `autor_id` aponte para um autor real. Você vai criar uma versão bem modelada e migrar os dados.

**Tarefa.**
1. Crie uma tabela `livros_catalogo` com: `id`; `titulo`; `autor_id` como `FOREIGN KEY` referenciando `autores(id)`; `ano_publicacao`; `exemplares_disponiveis`.
2. Migre **todos** os registros de `livros` para `livros_catalogo`.


---


In [17]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

_dntk.execute_sql(
  '-- Exercício 2\n-- Escreva sua solução SQL aqui.\nCREATE TABLE livros_catalogo (\n    id INTEGER PRIMARY KEY,\n    titulo VARCHAR(255) NOT NULL,\n    autor_id INTEGER NOT NULL,\n    ano_publicacao INTEGER,\n    exemplares_disponiveis INTEGER,\n    FOREIGN KEY (autor_id) REFERENCES autores(id)\n);\n\nINSERT INTO livros_catalogo (id, titulo, autor_id, ano_publicacao, exemplares_disponiveis)\nSELECT id, titulo, autor_id, ano_publicacao, exemplares_disponiveis\nFROM livros;',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)

,Count
0,11


### Exercício 3

A coordenação pediu uma listagem do acervo identificando quem escreveu cada livro.

**Tarefa.** Escreva uma única query com `INNER JOIN` entre `livros` e `autores` retornando `titulo`, `nome` do autor e `ano_publicacao`. Ordene por nome do autor (A→Z) e depois por ano de publicação (mais antigo → mais recente).


---


In [19]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

_dntk.execute_sql(
  '-- Exercício 3\n-- Escreva sua solução SQL aqui.\nSELECT livros.titulo, autores.nome, livros.ano_publicacao\nFROM livros\nINNER JOIN autores ON livros.autor_id = autores.id\nORDER BY autores.nome ASC, livros.ano_publicacao ASC;',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)

,titulo,nome,ano_publicacao
0,Americanah,Chimamanda Adichie,2013
1,Água Viva,Clarice Lispector,1973
2,A Hora da Estrela,Clarice Lispector,1977
3,Cem Anos de Solidão,Gabriel García Márquez,1967
4,O Amor nos Tempos do Cólera,Gabriel García Márquez,1985
5,Norwegian Wood,Haruki Murakami,1987
6,Capitães da Areia,Jorge Amado,1937
7,"Gabriela, Cravo e Canela",Jorge Amado,1958
8,Memórias Póstumas de Brás Cubas,Machado de Assis,1881
9,Quincas Borba,Machado de Assis,1891


### Exercício 4

A biblioteca quer identificar **livros que nunca foram emprestados** (candidatos a desativação do acervo).

**Tarefa.** Use a técnica de *anti-join*: `LEFT JOIN` entre `livros` e `emprestimos`, mantendo apenas as linhas em que **não há empréstimo correspondente**. Retorne `titulo` e `nome` do autor de cada livro nunca emprestado. 


---


In [21]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

_dntk.execute_sql(
  '-- Exercício 4\n-- Escreva sua solução SQL aqui.\nSELECT livros.titulo, autores.nome\nFROM livros\nINNER JOIN autores ON livros.autor_id = autores.id\nLEFT JOIN emprestimos ON livros.id = emprestimos.livro_id\nWHERE emprestimos.id IS NULL;',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)

,titulo,nome
0,Quincas Borba,Machado de Assis
1,"Gabriela, Cravo e Canela",Jorge Amado


### Exercício 5 · ⚠️ modifica dados

A biblioteca vai registrar o **gênero literário** dos livros e cadastrar **duas aquisições novas**.

**Tarefa.**
1. `ALTER TABLE livros` adicionando uma coluna de texto chamada `genero` .
2. Com `UPDATE` e `WHERE`, defina: `'Dom Casmurro'` → `'Romance'`; `'A Hora da Estrela'` → `'Romance'`; `'Capitães da Areia'` → `'Romance Social'`.
3. **Insira dois livros novos** na tabela `livros` (ids 12 e 13), de autores já existentes, já preenchendo o `genero`:
   - id 12, `'A Paixão Segundo G.H.'`, autor_id 2, ano 1964, 2 exemplares, gênero `'Romance'`.
   - id 13, `'Kafka à Beira-Mar'`, autor_id 5, ano 2002, 1 exemplar, gênero `'Fantasia'`.
4. `SELECT` retornando `titulo` e `genero` de todos os livros com gênero preenchido.

---


In [25]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

_dntk.execute_sql(
  '-- Exercício 5\n-- Escreva sua solução SQL aqui.\nALTER TABLE livros ADD COLUMN genero VARCHAR(50);\n\nUPDATE livros SET genero = \'Romance\' WHERE titulo = \'Dom Casmurro\';\nUPDATE livros SET genero = \'Romance\' WHERE titulo = \'A Hora da Estrela\';\nUPDATE livros SET genero = \'Romance Social\' WHERE titulo = \'Capitães da Areia\';\n\nINSERT INTO livros (id, titulo, autor_id, ano_publicacao, exemplares_disponiveis, genero) VALUES\n(12, \'A Paixão Segundo G.H.\', 2, 1964, 2, \'Romance\'),\n(13, \'Kafka à Beira-Mar\', 5, 2002, 1, \'Fantasia\');\n\nSELECT titulo, genero \nFROM livros \nWHERE genero IS NOT NULL;',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)

,titulo,genero
0,Dom Casmurro,Romance
1,A Hora da Estrela,Romance
2,Capitães da Areia,Romance Social
3,A Paixão Segundo G.H.,Romance
4,Kafka à Beira-Mar,Fantasia


### Exercício 6· ⚠️ modifica dados

A direção decidiu **reajustar as multas em 50%** para empréstimos ainda em aberto e que já têm multa.

**Tarefa.** Uma única query `UPDATE` que multiplique `valor_multa` por 1.5 apenas nas linhas em que `data_devolucao_real` é nula **e** `valor_multa` é maior que zero. Depois, `SELECT` retornando `id`, `livro_id`, `membro_id`, `valor_multa` dessas linhas.

**Saída.** Afeta 3 linhas:

| id | livro_id | membro_id | valor_multa |
|---|---|---|---|
| 4 | 2 | 3 | 22.5 |
| 5 | 8 | 4 | 42.0 |
| 6 | 10 | 2 | 15.0 |

---


In [27]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

_dntk.execute_sql(
  '-- Exercício 6\n-- Escreva sua solução SQL aqui.\nUPDATE emprestimos \nSET valor_multa = valor_multa * 1.5 \nWHERE data_devolucao_real IS NULL AND valor_multa > 0;\n\nSELECT id, livro_id, membro_id, valor_multa \nFROM emprestimos \nWHERE data_devolucao_real IS NULL AND valor_multa > 0;',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)

,id,livro_id,membro_id,valor_multa
0,4,2,3,23
1,5,8,4,42
2,6,10,2,15


# Banco 2 — Liga de Futebol

## Arquivos CSV do Banco 2 (o professor disponibiliza no projeto)

**`times.csv`**
```csv
id,nome
1,Flamengo
2,Palmeiras
3,Corinthians
4,Grêmio
```

**`jogadores.csv`**
```csv
id,nome,time_id
1,Pedro,1
2,Arrascaeta,1
3,Gabigol,1
4,Endrick,2
5,Veiga,2
6,Rony,2
7,Yuri Alberto,3
8,Cássio,3
9,Renato Augusto,3
10,Suárez,4
11,Diego Costa,4
12,Pepê,4
```

**`partidas.csv`**
```csv
id,time_mandante_id,time_visitante_id,gols_mandante,gols_visitante,data
1,1,2,2,1,2026-03-01
2,3,4,1,1,2026-03-08
3,1,3,3,0,2026-03-15
4,2,4,2,2,2026-03-22
5,2,3,1,0,2026-03-29
6,1,4,0,2,2026-04-05
7,1,2,3,1,2026-04-12
8,1,3,1,0,2026-04-19
9,4,3,2,1,2026-04-26
10,4,2,1,0,2026-05-03
```

**`gols.csv`**
```csv
id,partida_id,jogador_id,minuto
1,1,1,12
2,1,1,67
3,1,4,38
4,2,7,22
5,2,10,80
6,3,3,15
7,3,3,55
8,3,2,71
9,4,5,18
10,4,5,60
11,4,11,33
12,4,11,88
13,5,6,25
14,6,10,41
15,6,11,76
16,7,1,9
17,7,3,44
18,7,2,73
19,7,4,81
20,8,1,52
21,9,10,19
22,9,10,66
23,9,7,30
24,10,11,28
```


## Célula de importação do Banco 2 — *bloco SQL · execute antes dos exercícios 7 a 12*



---


In [29]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

_dntk.execute_sql(
  '-- IMPORTAÇÃO / RESET — Banco 2: Liga de Futebol\nCREATE OR REPLACE TABLE times     AS SELECT * FROM read_csv_auto(\'times.csv\');\nCREATE OR REPLACE TABLE jogadores AS SELECT * FROM read_csv_auto(\'jogadores.csv\');\nCREATE OR REPLACE TABLE partidas  AS SELECT * FROM read_csv_auto(\'partidas.csv\');\nCREATE OR REPLACE TABLE gols      AS SELECT * FROM read_csv_auto(\'gols.csv\');\n',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)

,Count
0,24


### Exercício 7

A redação esportiva quer um relatório detalhado de todos os gols, mostrando quem marcou e os dois times de cada partida.

**Tarefa.** Uma única query com o join adequado entre as tabelas necessárias retornando, por gol: nome do jogador, nome do time do jogador, nome do time mandante, nome do time visitante e minuto. Use **alias** na tabela `times` para referenciá-la três vezes (time do jogador, mandante, visitante). Ordene por `data` e depois por `minuto`.

---


In [31]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

_dntk.execute_sql(
  '-- Exercício 7\n-- Escreva sua solução SQL aqui.\nSELECT \n    jogadores.nome AS nome_jogador,\n    t_jogador.nome AS time_jogador,\n    t_mandante.nome AS time_mandante,\n    t_visitante.nome AS time_visitante,\n    gols.minuto\nFROM gols\nINNER JOIN jogadores ON gols.jogador_id = jogadores.id\nINNER JOIN partidas ON gols.partida_id = partidas.id\nINNER JOIN times AS t_jogador ON jogadores.time_id = t_jogador.id\nINNER JOIN times AS t_mandante ON partidas.time_mandante_id = t_mandante.id\nINNER JOIN times AS t_visitante ON partidas.time_visitante_id = t_visitante.id\nORDER BY partidas.data ASC, gols.minuto ASC;',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)

,nome_jogador,time_jogador,time_mandante,time_visitante,minuto
0,Pedro,Flamengo,Flamengo,Palmeiras,12
1,Endrick,Palmeiras,Flamengo,Palmeiras,38
2,Pedro,Flamengo,Flamengo,Palmeiras,67
3,Yuri Alberto,Corinthians,Corinthians,Grêmio,22
4,Suárez,Grêmio,Corinthians,Grêmio,80
5,Gabigol,Flamengo,Flamengo,Corinthians,15
6,Gabigol,Flamengo,Flamengo,Corinthians,55
7,Arrascaeta,Flamengo,Flamengo,Corinthians,71
8,Veiga,Palmeiras,Palmeiras,Grêmio,18
9,Diego Costa,Grêmio,Palmeiras,Grêmio,33


### Exercício 8

A comissão técnica quer um relatório **completo** dos jogadores, incluindo os que ainda não marcaram.

**Tarefa.** o join adequado entre as tabelas necessárias, agrupando por jogador e contando gols, incluindo **todos os 12 jogadores** (0 para quem não marcou). Ordene por total de gols decrescente, desempate por nome. 

---


In [33]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

_dntk.execute_sql(
  '-- Exercício 8\n-- Escreva sua solução SQL aqui.\nSELECT \n    jogadores.nome,\n    COUNT(gols.id) AS total_gols\nFROM jogadores\nLEFT JOIN gols ON jogadores.id = gols.jogador_id\nGROUP BY jogadores.id, jogadores.nome\nORDER BY total_gols DESC, jogadores.nome ASC;',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)

,nome,total_gols
0,Diego Costa,4
1,Pedro,4
2,Suárez,4
3,Gabigol,3
4,Arrascaeta,2
5,Endrick,2
6,Veiga,2
7,Yuri Alberto,2
8,Rony,1
9,Cássio,0


### Exercício 9

A organização quer mapear **duplas de companheiros de time** para uma campanha de marketing.

**Tarefa.** Use o join adequado para juntas a tabela `jogadores` com ela mesma e formar pares de jogadores do **mesmo time**. Evite duplicatas e auto-pares. Retorne nome do time, nome do jogador 1 e nome do jogador 2, ordenado por time.

---


In [35]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

_dntk.execute_sql(
  '-- Exercício 9\n-- Escreva sua solução SQL aqui.\nSELECT \n    times.nome AS nome_time,\n    j1.nome AS jogador_1,\n    j2.nome AS jogador_2\nFROM jogadores j1\nINNER JOIN jogadores j2 ON j1.time_id = j2.time_id AND j1.id < j2.id\nINNER JOIN times ON j1.time_id = times.id\nORDER BY times.nome ASC;',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)

,nome_time,jogador_1,jogador_2
0,Corinthians,Cássio,Renato Augusto
1,Corinthians,Yuri Alberto,Cássio
2,Corinthians,Yuri Alberto,Renato Augusto
3,Flamengo,Arrascaeta,Gabigol
4,Flamengo,Pedro,Arrascaeta
5,Flamengo,Pedro,Gabigol
6,Grêmio,Diego Costa,Pepê
7,Grêmio,Suárez,Diego Costa
8,Grêmio,Suárez,Pepê
9,Palmeiras,Veiga,Rony


### Exercício 10

Antes da fase eliminatória, a comissão quer o **saldo de gols** de cada time (gols pró menos gols contra), considerando todas as partidas.

**Tarefa.** Cada time aparece em partidas tanto como mandante quanto como visitante. Una `times` com `partidas` e use calcule:
- **gols pró** — `gols_mandante` quando o time é mandante; `gols_visitante` quando é visitante;
- **gols contra** — o inverso;
- **saldo** — pró menos contra.

Retorne `nome`, `gols_pro`, `gols_contra`, `saldo`, ordenado por saldo decrescente.

**Saída.** 4 linhas:

| nome | gols_pro | gols_contra | saldo |
|---|---|---|---|
| Flamengo | 9 | 4 | 5 |
| Grêmio | 8 | 4 | 4 |
| Palmeiras | 5 | 8 | -3 |
| Corinthians | 2 | 8 | -6 |

---


In [37]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

_dntk.execute_sql(
  '-- Exercício 10\n-- Escreva sua solução SQL aqui.\nSELECT \n    times.nome,\n    SUM(CASE WHEN times.id = partidas.time_mandante_id THEN partidas.gols_mandante ELSE partidas.gols_visitante END) AS gols_pro,\n    SUM(CASE WHEN times.id = partidas.time_mandante_id THEN partidas.gols_visitante ELSE partidas.gols_mandante END) AS gols_contra,\n    SUM(CASE WHEN times.id = partidas.time_mandante_id THEN partidas.gols_mandante ELSE partidas.gols_visitante END) - \n    SUM(CASE WHEN times.id = partidas.time_mandante_id THEN partidas.gols_visitante ELSE partidas.gols_mandante END) AS saldo\nFROM times\nINNER JOIN partidas ON times.id = partidas.time_mandante_id OR times.id = partidas.time_visitante_id\nGROUP BY times.id, times.nome\nORDER BY saldo DESC;',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)

,nome,gols_pro,gols_contra,saldo
0,Flamengo,9.0,4.0,5.0
1,Grêmio,8.0,4.0,4.0
2,Palmeiras,5.0,8.0,-3.0
3,Corinthians,2.0,8.0,-6.0


### Exercício 11

A query abaixo deveria retornar os **artilheiros da temporada** (jogadores com mais de 1 gol) com o nome do time, mas está com defeitos.

**Query quebrada (ponto de partida)**

```sql
SELECT j.nome, COUNT(*) AS gols, t.nome
FROM jogadores j
LEFT JOIN gols g ON g.jogador_id = j.id
JOIN times t ON j.time_id = t.id
WHERE COUNT(*) > 1
GROUP BY j.nome
ORDER BY gols;
```

**Tarefa.** Identifique **todos** os defeitos, corrija a query e produza uma versão funcional que retorne nome do jogador, nome do time e total de gols, apenas para jogadores com mais de 1 gol, ordenado do maior para o menor. Em comentários SQL no início, explique cada defeito e por que sua correção resolve.

---


In [ ]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

_dntk.execute_sql(
  '-- Exercício 11\n-- Escreva sua solução SQL aqui.\n\'\'\'\nDefeitos identificados e correções aplicadas:\n1. Uso do COUNT(*) no WHERE: Funções de agregação não podem ser usadas na cláusula WHERE porque ela filtra as linhas antes do agrupamento. Foi corrigido movendo a condição para a cláusula HAVING.\n2. Agrupamento incompleto: A query original agrupa apenas por j.nome. No SQL padrão, todas as colunas que não estão em funções de agregação (como t.nome) devem estar no GROUP BY. Foi adicionado t.nome ao agrupamento.\n3. Ordenação incorreta: A ordenação original estava em ordem crescente (padrão ASC). Como o objetivo é listar os artilheiros, foi adicionado o modificador DESC para ordenar do maior para o menor total de gols.\n\'\'\'\n\nSELECT j.nome AS nome_jogador, t.nome AS nome_time, COUNT(g.id) AS gols\nFROM jogadores j\nINNER JOIN gols g ON g.jogador_id = j.id\nINNER JOIN times t ON j.time_id = t.id\nGROUP BY j.nome, t.nome\nHAVING COUNT(g.id) > 1\nORDER BY gols DESC;',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)

### Exercício 12 · ⚠️ modifica dados

Uma nova rodada foi disputada e precisa ser registrada no banco.

**Tarefa.**
1. **Insira** em `partidas` a partida de id 11: mandante Palmeiras (2), visitante Corinthians (3), 2×1, data `'2026-05-10'`.
2. **Insira** em `gols` os três gols dessa partida: jogador 5 (Veiga) aos 20', jogador 4 (Endrick) aos 55', jogador 7 (Yuri Alberto) aos 70'. Use ids de gol 25, 26 e 27.
3. Escreva uma query que confirme a inserção: liste os gols da partida 11 com nome do jogador e minuto (`INNER JOIN` entre `gols` e `jogadores`, filtrando `partida_id = 11`).

**Saída.** 3 linhas: Veiga (20), Endrick (55), Yuri Alberto (70). Demonstra que os novos dados foram inseridos corretamente.

---


In [39]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

_dntk.execute_sql(
  '-- Exercício 12\n-- Escreva sua solução SQL aqui.\nINSERT INTO partidas (id, time_mandante_id, time_visitante_id, gols_mandante, gols_visitante, data) VALUES\n(11, 2, 3, 2, 1, \'2026-05-10\');\n\nINSERT INTO gols (id, partida_id, jogador_id, minuto) VALUES\n(25, 11, 5, 20),\n(26, 11, 4, 55),\n(27, 11, 7, 70);\n\nSELECT jogadores.nome, gols.minuto\nFROM gols\nINNER JOIN jogadores ON gols.jogador_id = jogadores.id\nWHERE gols.partida_id = 11;',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)

,nome,minuto
0,Endrick,55
1,Veiga,20
2,Yuri Alberto,70


# 🅑 Parte B — Projeto de Modelagem · Marketplace Artesanal

Esta parte é resolvida **nas células finais deste notebook**. Ela mistura **células de texto (Markdown)** — onde você escreve suas análises — e **células SQL** — onde você implementa e testa o schema. 

Diferente da Parte A (foco em execução), aqui o foco é **projetar**. Leia o cenário, importe a planilha desnormalizada para inspecioná-la, e construa um modelo relacional correto.

## A planilha atual — arquivo CSV (o professor disponibiliza no projeto)

**`pedidos_planilha.csv`** (uma linha por item de pedido; dados de cliente e pedido repetem entre linhas; a coluna `categorias` empacota vários valores separados por `;`)
```csv
pedido_id,data,cliente,cli_email,cli_cidade,vendedor,vend_email,vend_pix,produto,preco_unit,qtd,categorias
P001,2026-04-10,Marina Costa,marina@mail.com,São Paulo,Atelier Lua,lua@atelier.com.br,lua-pix-01,Vaso de cerâmica azul,85.00,2,"decoração; cerâmica"
P001,2026-04-10,Marina Costa,marina@mail.com,São Paulo,Tecidos Nelm,nelm@tecidos.com.br,nelm-pix-02,Almofada bordada flores,45.00,1,"casa; têxtil; decoração"
P002,2026-04-11,Tiago Reis,tiago@mail.com,Curitiba,Atelier Lua,lua@atelier.com.br,lua-pix-01,Vaso de cerâmica azul,85.00,1,"decoração; cerâmica"
P003,2026-04-12,Marina Costa,marina@mail.com,São Paulo,Madeira Viva,madv@artesa.com.br,madv-pix-03,Porta-copos rústico (kit 4),60.00,1,"casa; madeira"
P003,2026-04-12,Marina Costa,marina@mail.com,São Paulo,Tecidos Nelm,nelm@tecidos.com.br,nelm-pix-02,Almofada bordada flores,45.00,2,"casa; têxtil; decoração"
P004,2026-04-15,Larissa Veiga,lari@mail.com,Recife,Madeira Viva,madv@artesa.com.br,madv-pix-03,Tábua de queijo entalhada,110.00,1,"cozinha; madeira"
```


## Célula de importação da planilha — *bloco SQL*


In [41]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

_dntk.execute_sql(
  '-- IMPORTAÇÃO — Planilha denormalizada do Marketplace\nCREATE OR REPLACE TABLE pedidos_planilha AS SELECT * FROM read_csv_auto(\'pedidos_planilha.csv\');\nSELECT * FROM pedidos_planilha;',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)

,pedido_id,data,cliente,cli_email,cli_cidade,vendedor,vend_email,vend_pix,produto,preco_unit,qtd,categorias
0,P001,2026-04-10,Marina Costa,marina@mail.com,São Paulo,Atelier Lua,lua@atelier.com.br,lua-pix-01,Vaso de cerâmica azul,85.0,2,decoração; cerâmica
1,P001,2026-04-10,Marina Costa,marina@mail.com,São Paulo,Tecidos Nelm,nelm@tecidos.com.br,nelm-pix-02,Almofada bordada flores,45.0,1,casa; têxtil; decoração
2,P002,2026-04-11,Tiago Reis,tiago@mail.com,Curitiba,Atelier Lua,lua@atelier.com.br,lua-pix-01,Vaso de cerâmica azul,85.0,1,decoração; cerâmica
3,P003,2026-04-12,Marina Costa,marina@mail.com,São Paulo,Madeira Viva,madv@artesa.com.br,madv-pix-03,Porta-copos rústico (kit 4),60.0,1,casa; madeira
4,P003,2026-04-12,Marina Costa,marina@mail.com,São Paulo,Tecidos Nelm,nelm@tecidos.com.br,nelm-pix-02,Almofada bordada flores,45.0,2,casa; têxtil; decoração
5,P004,2026-04-15,Larissa Veiga,lari@mail.com,Recife,Madeira Viva,madv@artesa.com.br,madv-pix-03,Tábua de queijo entalhada,110.0,1,cozinha; madeira


## Cenário

A startup **Feito à Mão Marketplace** controla seus pedidos na planilha acima. Com o crescimento, ela virou uma bagunça: dados duplicados, células com múltiplos valores e dificuldade para consultar quem vendeu o quê. Você foi contratado para projetar a base relacional que vai substituí-la.

## Regras de negócio (cada uma implica uma decisão de modelagem)

1. **Um cliente pode fazer vários pedidos**, mas cada pedido pertence a um único cliente.
2. **Um pedido pode conter itens de vendedores diferentes** (veja P001 e P003).
3. **Um produto pertence a um único vendedor** — não há produtos compartilhados.
4. **Um produto pode estar em uma ou mais categorias**, e uma categoria pode ter vários produtos.
5. **Cada vendedor possui uma única conta bancária para recebimentos** (com a chave Pix), e essa conta pertence exclusivamente a ele.
6. **O email de cliente e de vendedor identifica a pessoa de forma única** — não pode haver dois cadastros com o mesmo email.
7. **O preço de um produto pode mudar com o tempo**, mas o pedido deve preservar o preço cobrado **no momento da compra**.
8. **Categorias são poucas e padronizadas** — devem viver numa tabela própria, não como texto solto.

---


## ✍️ Exercício 13 — Reflexão e análise de entidades

Escreva nesta célula:

- **Reflexão inicial (~10 linhas):** por que um banco relacional é apropriado aqui? Que problemas concretos a planilha já mostra que um bom modelo resolve? *(Cobre a base histórica/conceitual: por que relacional, vantagens de boas práticas.)*
- **Entidades:** liste cada entidade identificada (ex: Cliente, Vendedor, Conta Bancária, Produto, Categoria, Pedido, Item de Pedido), com seus atributos e a justificativa de por que é entidade própria — e não atributo de outra.

---


## Resposta — Exercício 13

### O modelo de planilha atual apresenta graves problemas de integridade e redundância de dados (anomalias de inserção, atualização e exclusão). Se a cliente Marina Costa mudar de cidade, o sistema exigirá múltiplos updates em linhas diferentes, gerando o risco de inconsistência. Além disso, a coluna categorias viola a Primeira Forma Normal (1FN) ao armazenar múltiplos valores em uma única célula, o que inviabiliza buscas eficientes e agregações precisas.

### Um Banco de Dados Relacional é ideal para este cenário porque substitui a redundância pela consistência por meio do processo de normalização. Ao fragmentar a planilha em tabelas logicamente interligadas por chaves estrangeiras, garantimos a unicidade dos cadastros (clientes e vendedores únicos por e-mail), a preservação histórica de valores (como o preço de venda no momento do pedido) e a flexibilidade para consultas complexas (como descobrir o faturamento por categoria). A integridade referencial impede que um item de pedido aponte para um produto inexistente, blindando as regras de negócio da startup contra falhas humanas ou operacionais.


### Entidades Identificadas

**ClientesAtributos:** id, nome, email (único), cidade
**Justificativa:** Possui existência independente e ciclo de vida próprio anterior às compras. Centraliza os dados cadastrais do comprador sem a necessidade de replicá-los a cada novo item ou pedido realizado.

**VendedoresAtributos:** id, nome, email (único), pix
**Justificativa:** Entidade independente que centraliza os dados de contato e faturamento do artesão. Como cada vendedor possui uma única chave Pix exclusiva, ela atua como um atributo direto desta estrutura.

**ProdutosAtributos:** id, vendedor_id (FK), nome, preco_atual
**Justificativa:** Representa os itens físicos do catálogo. Pertence a um vendedor específico ($1:N$) e existe independentemente de ter sido comprado ou não, contendo o preço base atual praticado pelo artesão.

**CategoriasAtributos:** id, nome (único)
**Justificativa:** Garante a padronização e o reaproveitamento das classificações do catálogo (ex: "decoração"). Isolar em uma tabela própria evita erros de digitação e elimina textos redundantes no banco.

**Produtos_Categorias (Entidade Associativa)Atributos:** produto_id (FK), categoria_id (FK)
**Justificativa:** Resolve o relacionamento de muitos-para-muitos ($N:M$) entre produtos e categorias, permitindo que um item receba múltiplas tags e uma tag pertença a vários itens.

**PedidosAtributos:** id, cliente_id (FK), data
**Justificativa:** Registra o cabeçalho de uma transação comercial unificada feita por um cliente em uma data específica. Consolida o vínculo inicial de propriedade da compra antes do detalhamento dos itens.

**Itens_PedidosAtributos:** pedido_id (FK), produto_id (FK), quantidade, preco_pago
**Justificativa:** Resolve a relação muitos-para-muitos ($N:M$) entre pedidos e produtos. É fundamental para armazenar a quantidade exata comprada e congelar o preco_pago histórico no momento da venda.


## ✍️ Exercício 14 — Relacionamentos e chaves 
Escreva nesta célula um diagrama textual de relacionamentos. Para cada relação entre entidades, indique: entidades envolvidas, cardinalidade (1:1, 1:N, N:N) e como será implementada (chave estrangeira em qual lado, ou tabela de ligação). Garanta cobrir explicitamente:

- o **1:1** entre vendedor e conta bancária;
- os **1:N** (vendedor → produto, cliente → pedido, pedido → item de pedido);
- o **N:N** entre produto e categoria;
- onde o **preço do momento da compra** é registrado (regra 7).

---


## Resposta — Exercício 14

## Diagrama Textual de Relacionamentos

### 1. Vendedor — Conta Bancária

* **Entidades:** Vendedor, ContaBancaria
* **Cardinalidade:** 1:1
* **Implementação:**

  * A tabela `ContaBancaria` possui uma chave estrangeira `vendedor_id` referenciando `Vendedor`.
  * Deve haver uma restrição `UNIQUE` em `vendedor_id` para garantir o relacionamento 1:1.

---

### 2. Vendedor — Produto

* **Entidades:** Vendedor, Produto
* **Cardinalidade:** 1:N (um vendedor possui vários produtos)
* **Implementação:**

  * A tabela `Produto` contém a chave estrangeira `vendedor_id` apontando para `Vendedor`.

---

### 3. Cliente — Pedido

* **Entidades:** Cliente, Pedido
* **Cardinalidade:** 1:N (um cliente pode realizar vários pedidos)
* **Implementação:**

  * A tabela `Pedido` possui a chave estrangeira `cliente_id` referenciando `Cliente`.

---

### 4. Pedido — Item de Pedido

* **Entidades:** Pedido, ItemPedido
* **Cardinalidade:** 1:N (um pedido contém vários itens)
* **Implementação:**

  * A tabela `ItemPedido` possui a chave estrangeira `pedido_id` apontando para `Pedido`.

---

### 5. Produto — Categoria

* **Entidades:** Produto, Categoria
* **Cardinalidade:** N:N
* **Implementação:**

  * Criar uma tabela de ligação `ProdutoCategoria` com os campos:

    * `produto_id` (FK para `Produto`)
    * `categoria_id` (FK para `Categoria`)
  * Chave primária composta (`produto_id`, `categoria_id`)

---

### Regra 7 — Preço no momento da compra

* O preço do produto não deve ser obtido diretamente da tabela `Produto`, pois pode variar ao longo do tempo.
* **Implementação:**

  * Armazenar o preço na tabela `ItemPedido`.
* **Campo:**

  * `preco_unitario`
* **Objetivo:**

  * Garantir o registro histórico correto do valor pago no momento da compra.


## ✍️ Exercício 15 — Análise de normalização *(célula de texto / Markdown)*

Escreva nesta célula. Aponte **pelo menos uma violação** da planilha original em cada nível e mostre como seu modelo resolve:

- **1FN** — qual coluna guarda múltiplos valores numa só célula?
- **2FN** — qual atributo depende só de parte de uma chave composta?
- **3FN** — qual atributo depende de outro atributo não-chave (dependência transitiva)?

Cite as colunas concretas da planilha em cada caso.

---


## Resposta — Exercício 15

### Análise de Normalização

* **1FN (Primeira Forma Normal)**
    * **Violação na Planilha:** A coluna `categorias` guarda múltiplos valores separados por ponto e vírgula na mesma célula (ex: `"decoração; cerâmica"` na linha do Vaso de cerâmica). Isso viola a regra de que todos os valores devem ser atômicos.
    * **Solução no Modelo:** Criação de uma tabela exclusiva para `categorias` e de uma tabela associativa `produtos_categorias` ($N:M$), onde cada linha armazena estritamente um `produto_id` e um `categoria_id`.

---

* **2FN (Segunda Forma Normal)**
    * **Violação na Planilha:** Se pensarmos na planilha original como uma grande tabela de itens onde a chave primária lógica seria composta por (`pedido_id`, `produto`), os atributos do cliente (`cliente`, `cli_email`, `cli_cidade`) dependem apenas de parte da chave (`pedido_id`), e não do produto. Da mesma forma, os dados do vendedor dependem apenas do produto. Isso caracteriza dependência parcial.
    * **Solução no Modelo:** Os dados foram divididos em entidades com chaves simples. Os atributos do cliente foram para a tabela `clientes` (onde dependem exclusivamente de `cliente_id`), e os dados do pedido ficaram em `pedidos` (dependendo apenas de `pedido_id`).

---

* **3FN (Terceira Forma Normal)**
    * **Violação na Planilha:** O atributo `cli_cidade` depende diretamente de `cli_email` (ou do cliente em si), que por sua vez depende de `pedido_id`. Existe aqui uma dependência transitiva: $\text{pedido\_id} \rightarrow \text{cli\_email} \rightarrow \text{cli\_cidade}$. O mesmo ocorre com `vend_pix` dependendo de `vendedor`, que dependia do produto.
    * **Solução no Modelo:** Eliminação da transitividade ao isolar os atributos em suas respectivas tabelas. Agora, `cidade` é um atributo que depende única e diretamente da chave primária em `clientes`, e `pix` depende diretamente da chave em `vendedores`.


## 💾 Exercício 16 — Implementação física *(células SQL)*

Nas células SQL finais, implemente seu modelo:

1. Todos os `CREATE TABLE` do modelo final, com tipos, `PRIMARY KEY`, `FOREIGN KEY`, e `UNIQUE` / `NOT NULL` conforme as regras de negócio (atenção especial ao `UNIQUE` dos emails e ao lado 1:1).
2. `INSERT` reproduzindo os **6 registros da planilha** no formato normalizado. Se algum dado da planilha não tem onde caber no seu modelo, o modelo está incompleto.
3. Uma query final de verificação à sua escolha (por exemplo, um `JOIN` que reconstrói uma linha original da planilha a partir das tabelas normalizadas), demonstrando que nenhuma informação foi perdida.

> Como estas são células SQL executáveis, seu schema **roda de verdade**: se um `CREATE TABLE` ou `INSERT` tiver erro, o Deepnote vai acusar. Aproveite isso para validar antes de entregar.

---


In [45]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

_dntk.execute_sql(
  '-- Exercício 16 — CREATE TABLE do modelo final\n-- Escreva aqui os comandos CREATE TABLE com PK, FK, UNIQUE e NOT NULL.\nCREATE TABLE clientes (\n    id INTEGER PRIMARY KEY,\n    nome VARCHAR(100) NOT NULL,\n    email VARCHAR(100) NOT NULL UNIQUE,\n    cidade VARCHAR(100) NOT NULL\n);\n\nCREATE TABLE vendedores (\n    id INTEGER PRIMARY KEY,\n    nome VARCHAR(100) NOT NULL,\n    email VARCHAR(100) NOT NULL UNIQUE,\n    pix VARCHAR(100) NOT NULL UNIQUE\n);\n\nCREATE TABLE produtos (\n    id INTEGER PRIMARY KEY,\n    vendedor_id INTEGER NOT NULL,\n    nome VARCHAR(150) NOT NULL,\n    preco_atual DECIMAL(10, 2) NOT NULL,\n    FOREIGN KEY (vendedor_id) REFERENCES vendedores(id)\n);\n\nCREATE TABLE categorias (\n    id INTEGER PRIMARY KEY,\n    nome VARCHAR(50) NOT NULL UNIQUE\n);\n\nCREATE TABLE produtos_categorias (\n    produto_id INTEGER NOT NULL,\n    categoria_id INTEGER NOT NULL,\n    PRIMARY KEY (produto_id, categoria_id),\n    FOREIGN KEY (produto_id) REFERENCES produtos(id),\n    FOREIGN KEY (categoria_id) REFERENCES categorias(id)\n);\n\nCREATE TABLE pedidos (\n    id VARCHAR(10) PRIMARY KEY,\n    cliente_id INTEGER NOT NULL,\n    data DATE NOT NULL,\n    FOREIGN KEY (cliente_id) REFERENCES clientes(id)\n);\n\nCREATE TABLE itens_pedidos (\n    pedido_id VARCHAR(10) NOT NULL,\n    produto_id INTEGER NOT NULL,\n    quantidade INTEGER NOT NULL,\n    preco_pago DECIMAL(10, 2) NOT NULL,\n    PRIMARY KEY (pedido_id, produto_id),\n    FOREIGN KEY (pedido_id) REFERENCES pedidos(id),\n    FOREIGN KEY (produto_id) REFERENCES produtos(id)\n);',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)

,Count


In [47]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

_dntk.execute_sql(
  '-- Exercício 16 — INSERT dos dados normalizados\n-- Insira aqui os dados que reproduzem os 6 registros da planilha original.\nINSERT INTO clientes (id, nome, email, cidade) VALUES \n(1, \'Marina Costa\', \'marina@mail.com\', \'São Paulo\'),\n(2, \'Tiago Reis\', \'tiago@mail.com\', \'Curitiba\'),\n(3, \'Larissa Veiga\', \'lari@mail.com\', \'Recife\');\n\n\nINSERT INTO vendedores (id, nome, email, pix) VALUES \n(1, \'Atelier Lua\', \'lua@atelier.com.br\', \'lua-pix-01\'),\n(2, \'Tecidos Nelm\', \'nelm@tecidos.com.br\', \'nelm-pix-02\'),\n(3, \'Madeira Viva\', \'madv@artesa.com.br\', \'madv-pix-03\');\n\n\nINSERT INTO produtos (id, vendedor_id, nome, preco_atual) VALUES \n(1, 1, \'Vaso de cerâmica azul\', 85.00),\n(2, 2, \'Almofada bordada flores\', 45.00),\n(3, 3, \'Porta-copos rústico (kit 4)\', 60.00),\n(4, 3, \'Tábua de queijo entalhada\', 110.00);\n\n\nINSERT INTO categorias (id, nome) VALUES \n(1, \'decoração\'),\n(2, \'cerâmica\'),\n(3, \'casa\'),\n(4, \'têxtil\'),\n(5, \'madeira\'),\n(6, \'cozinha\');\n\n\nINSERT INTO produtos_categorias (produto_id, categoria_id) VALUES \n(1, 1), (1, 2), -- Vaso: decoração, cerâmica\n(2, 3), (2, 4), (2, 1), -- Almofada: casa, têxtil, decoração\n(3, 3), (3, 5), -- Porta-copos: casa, madeira\n(4, 6), (4, 5); -- Tábua: cozinha, madeira\n\n\nINSERT INTO pedidos (id, cliente_id, data) VALUES \n(\'P001\', 1, \'2026-04-10\'),\n(\'P002\', 2, \'2026-04-11\'),\n(\'P003\', 1, \'2026-04-12\'),\n(\'P004\', 3, \'2026-04-15\');\n\n\nINSERT INTO itens_pedidos (pedido_id, produto_id, quantidade, preco_pago) VALUES \n(\'P001\', 1, 2, 85.00),\n(\'P001\', 2, 1, 45.00),\n(\'P002\', 1, 1, 85.00),\n(\'P003\', 3, 1, 60.00),\n(\'P003\', 2, 2, 45.00),\n(\'P004\', 4, 1, 110.00);',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)

,Count
0,6


In [49]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

_dntk.execute_sql(
  '-- Exercício 16 — Query final de verificação\n-- Escreva aqui um JOIN que demonstre que nenhuma informação foi perdida.\nSELECT \n    p.id AS pedido_id,\n    p.data,\n    c.nome AS cliente,\n    c.email AS cli_email,\n    c.cidade AS cli_cidade,\n    v.nome AS vendedor,\n    v.email AS vend_email,\n    v.pix AS vend_pix,\n    prod.nome AS produto,\n    ip.preco_pago AS preco_unit,\n    ip.quantidade AS qtd\nFROM itens_pedidos ip\nINNER JOIN pedidos p ON ip.pedido_id = p.id\nINNER JOIN clientes c ON p.cliente_id = c.id\nINNER JOIN produtos prod ON ip.produto_id = prod.id\nINNER JOIN vendedores v ON prod.vendedor_id = v.id\nORDER BY p.id, prod.nome;',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)

,pedido_id,data,cliente,cli_email,cli_cidade,vendedor,vend_email,vend_pix,produto,preco_unit,qtd
0,P001,2026-04-10,Marina Costa,marina@mail.com,São Paulo,Tecidos Nelm,nelm@tecidos.com.br,nelm-pix-02,Almofada bordada flores,45.0,1
1,P001,2026-04-10,Marina Costa,marina@mail.com,São Paulo,Atelier Lua,lua@atelier.com.br,lua-pix-01,Vaso de cerâmica azul,85.0,2
2,P002,2026-04-11,Tiago Reis,tiago@mail.com,Curitiba,Atelier Lua,lua@atelier.com.br,lua-pix-01,Vaso de cerâmica azul,85.0,1
3,P003,2026-04-12,Marina Costa,marina@mail.com,São Paulo,Tecidos Nelm,nelm@tecidos.com.br,nelm-pix-02,Almofada bordada flores,45.0,2
4,P003,2026-04-12,Marina Costa,marina@mail.com,São Paulo,Madeira Viva,madv@artesa.com.br,madv-pix-03,Porta-copos rústico (kit 4),60.0,1
5,P004,2026-04-15,Larissa Veiga,lari@mail.com,Recife,Madeira Viva,madv@artesa.com.br,madv-pix-03,Tábua de queijo entalhada,110.0,1


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=b38e2658-87e8-4c95-a56a-acf308dd67d3' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>